<a href="https://colab.research.google.com/github/hxk271/SocDataSci/blob/main/W10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 10 (자료의 전처리 II)

> 자료 전처리(data preprocessing)를 오늘도 계속 연습하기로 한다. 특히 오늘은 자료의 **집계(aggregation)**, **결합(merging)**, 그리고 **재배열(reshaping)**에 관해 공부한다.
<br><br>
> 본격적인 연습을 위해 다음 링크에서 오늘 사용할 파일들을 모두 준비하자.

In [ ]:
import gdown
links = ['https://drive.google.com/uc?id=1bXEgep5eQ__R2kOIHD5x2hP57IOAtktV',          #niaa2009
         'https://drive.google.com/uc?id=1bYVuWXqOhb_32ExO2tTTrS_lqKnLBNUp',          #division
         'https://drive.google.com/uc?id=1bYQAy0y4zMGxWRZ4jPvv2BLkMIunuuG-',          #niaaa-report
         'https://drive.google.com/uc?id=1bZC-t1hm5ybla7gEr44UiS4K8ILS0l1B',          #population
         'https://drive.google.com/uc?id=1WLQ60NVo0wtrtCRf1odO-VzLmZ7ebVao',          #lifeexp
         'https://drive.google.com/uc?id=1hsBgmMLiky5sXUSy-jt3ESsqNNyMERyr',          #titanic
         'https://drive.google.com/uc?id=1hyYA9GyOeRryug08f1h4Gb85lcO_PbLG',          #german-credit
         'https://drive.google.com/uc?id=1csKImQToC6_JTRbyan9aJuDq9UYHw4tr',          #uber
         'https://drive.google.com/uc?id=1hpr7M5QmNCUgDbLd-Y8OnnaBZNWZpc-r']          #penguins

for link in links:
    gdown.download(link)

## 1. 집계

> `niaa-report.csv`를 불러와 어떤 식으로 인덱스(index)를 설정하면 좋을지 살펴보자.

In [ ]:
import pandas as pd

alco = pd.read_csv("niaaa-report.csv", index_col = "State")
alco

alco = pd.read_csv("niaaa-report.csv", index_col = ['State', 'Year'])
alco

> 일단 수열이나 데이터프레임이 주어진다면, <code>max()</code>, <code>min()</code>, <code>mean()</code>, <code>sum()</code> 등으로 기초통계량을 쉽게 계산할 수 있다는 사실을 기억하자. 이때 `axis=0`를 사용하면 변수 간(across) 집계를 수행할 수 있고, `axis=1`를 사용하면 변수 내(within) 집계를 수행할 수 있다.


In [ ]:
#최대/최소
alco.max()
alco['Wine'].min()

#평균
alco.mean(axis = 1)
alco.median()

#합계
alco.sum()
alco.sum(axis = 1)

> 그런데 이런 계산을 전체에 대해 그리고 그룹별로 수행하면 곧 **집계(aggregation)**가 된다. 집계는 대단히 중요한데 제대로 수행하기 제법 어렵다.
<br><br>
>  데이터프레임에서 `groupby()`를 제대로 사용할 수 있어야 한다. 괄호 안에는 어떤 변수 단위로 집계할지 넣어준다.

In [ ]:
mean_alco = alco.groupby("Year").mean()
mean_alco

> **연습문제 1-1**. `niaaa-report.csv`에서 맥주 총소비량을 주별로 집계하시오.

In [ ]:
alco.groupby('State')['Beer'].mean()

> **연습문제 1-2**. `niaaa-report.csv`에서 세 종류 주류의 총소비량을 연도별로 집계하시오.

In [ ]:
alco['alc'] = alco['Beer'] + alco['Wine'] + alco['Spirits']
alco.groupby('Year')['alc'].sum()
alco

alco.groupby('Year')['alc'].sum().plot()

> **연습문제 1-3**. `penguins.csv`를 활용하여 펭귄의 종별로 부리(bill), 날개(flipper), 몸무게 평균을 계산하시오.

In [ ]:
df = pd.read_csv('penguins.csv')
df.groupby('species')[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']].mean()

> **연습문제 1-4**. `titanic.csv`를 활용하여 (1) 성별-좌석등급별 생존율, (2) 성별 총 생존자 수를 각각 계산하시오.

In [ ]:
df = pd.read_csv('titanic.csv')

#성별-좌석등급별 생존율
df.groupby(['sex','class'])['survived'].mean()

#성별 생존자수
df.groupby('sex')['survived'].sum()

> 조금 수준이 높긴 하지만 `agg()`를 사용하면 필요한 항목들을 한번에 계산할 수 있다.

In [ ]:
df.groupby(['sex','class'])[['survived','age']].agg({'survived' : 'mean', 'age' : 'max'})

## 2. 결합

> 때로 두 개 이상의 자료가 따로따로 떨어져 있을 때보다 하나로 합쳐졌을 때 큰 잠재력을 발휘하곤 한다. 자료를 합치기 위해서는 먼저 합칠 두 자료를 각각 불러오고 **공통 식별자(common identifier)**를 찾아야 한다.
<br><br>
> 두 개의 데이터를 불러와 각각 내용을 살펴보자.

In [ ]:
drink = pd.read_csv("niaaa-report2009.csv")
drink

pop = pd.read_csv("population.csv")
pop

> 만약 연도별 혹은 주별로 1인당 맥주 소비량을 계산하려면 어떻게 해야할까? 일단 하나의 자료만으로는 그런 계산을 수행할 수 없다(Why?). 두 자료를 결합해야만 한다.
<br><br>
> `pd.merge()`로 두 자료를 결합할 수 있다. 상당히 중요한 스킬이므로 잘 이해할 필요가 있다.

In [ ]:
df = pd.merge(drink, pop, left_on = "State", right_on = "State")       #왼쪽 오른쪽 각각의 공통 변수
df

> 만약 공통된 인덱스(common index)가 있다면 아주 쉽게 결합할 수 있다! 만일 다른 변수가 모두 숫자라면 나중에 계산을 편리하게 하기 위해 고유한 문자열(e.g., 주 이름)을 인덱스로 삼을 수도 있다. 이때는 <code>index_col</code> 옵션을 활용한다.

In [ ]:
drink = pd.read_csv("niaaa-report2009.csv")
drink = drink.set_index("State")
drink

pop = pd.read_csv("population.csv", index_col = "State")
pop

> 이때는 `pd.merge()`를 다른 방식의 패러미터와 함께 사용한다.

In [ ]:
df = pd.merge(drink, pop, left_index = True, right_index = True)
df

> 위에서 설명한 둘 중 하나의 방식으로 자료를 결합할 수 있다. 물론 혼합할 수도 있다. 이제 미국 주들의 인구 십만 명당 맥주 소비량을 계산해보자.

In [ ]:
df['beer_per_capita'] = df['Beer'] / df['Population']
df

> 안된다. 왜 그럴까? 사실 성급하게 결합하려고 했던 점에서 문제가 예견되었다. 사실 합치기 전에 자료를 먼저 꼼꼼하게 살펴보아야 한다! `pop` 자료를 먼저 꼼꼼히 살펴보고 인구별로 **정렬(sort)**도 해보자.

In [ ]:
pop.sort_values("Population")     #Population 변수별로 정렬

> 위의 결과가 이상하다. 하지만 잘 들여다보면 말이 되긴 한다. 근본적인 이유는 자료유형(dtype) 때문이다!

In [ ]:
pop.dtypes

> `Population`이 **객체(object)**로 설정되어 있으므로 숫자 계산에서 오류가 나타났다! 우리는 이 문제에 대응하는 방법을 이미 배웠다. 또다른 방법은 애시당초 자료를 불러올때부터 문제의 원인을 지목하는 것이다!

In [ ]:
pop = pd.read_csv("population.csv", index_col="State", thousands = ',')
pop.dtypes

pop.sort_values("Population", ascending = True)

> > 이제 다시 미국 주들의 인구 십만 명당 맥주 소비량을 계산해보자.

In [ ]:
df = pd.merge(drink, pop, left_on = "State", right_on = "State")

df['beer_per_capita'] = df['Beer'] / (df['Population'] / 100000)
df.sort_values('beer_per_capita', ascending = False)

> **연습문제 2-1**. `niaaa-report2009.csv`와 `division.csv`를 요령껏 결합하시오.

In [ ]:
import pandas as pd

alco2009 = pd.read_csv("niaaa-report2009.csv")
alco2009

division = pd.read_csv("division.csv")
division

df = pd.merge(alco2009, division, left_on = 'State', right_on = 'state')
df

> `pd.merge()` 이외의 방식도 가능하다. 각각 쓰임새가 조금씩 다르다. 가령 <code>pd.concat()</code>도 자주 쓰인다. 이때는 인덱스가 공통적으로 설정되어 있어야 한다.

In [ ]:
pop = pd.read_csv("population.csv", index_col="State", thousands = ',')
drink = pd.read_csv("niaaa-report2009.csv", index_col = "State")

df = pd.concat([pop, drink], axis = 1)
df

> `pd.concat()`을 사용하여 결합(merging)이 아니라, **추가(appending)**할 수도 있다. 이건 두 자료를 좌우로 합치는 게 아니라 위아래로 합치는 것인데 <code>pd.concat()</code>으로 할 수 있다.

In [ ]:
df = pd.concat([pop, drink], axis = 0)
df

> 두 개 이상의 수열이 합쳐지면 결국 하나의 행렬, 즉 자료가 된다. 그러므로 `pd.concat()`을 사용하여 수열을 결합(merging)한다면 하나의 데이터프레임으로 만들 수 있다!

> **연습문제 2-2**. 아래 연도(`year`), 실업률(`unemployment`), 인플레이션율(`inflation`) 수열을 하나로 합친 데이터프레임을 만드시오.
---
```python
year = pd.Series(range(2014, 2024))
unemployment = pd.Series([3.5, 3.6, 3.7, 3.7, 3.8, 3.8, 4.0, 3.7, 2.9, 2.7])
inflation = pd.Series([2.0, 2.2, 1.6, 1.5, 1.2, 0.9, 0.7, 1.8, 4.1, 4.0])
```

In [ ]:
year = pd.Series(range(2014, 2024))
unemployment = pd.Series([3.5, 3.6, 3.7, 3.7, 3.8, 3.8, 4.0, 3.7, 2.9, 2.7])
inflation = pd.Series([2.0, 2.2, 1.6, 1.5, 1.2, 0.9, 0.7, 1.8, 4.1, 4.0])

df = pd.concat([year, inflation, unemployment], axis = 1)     #axis=0도 실험해보자
df.columns = ["year", "inflation", "unemployment"]
df

## 3. 재배열

> 분석상 필요에 따라 자료의 꼴을 **재배열(reshape)**해야할 때가 있다. 이것도 자료 전처리 중에서도 상당히 까다로운 스킬이다.
>
> 여러 가지 방법으로 자료를 재배열할 수 있지만, 일단 인덱스를 먼저 설정해놓고 <code>stack()</code>과 <code>unstack()</code>을 사용하는 편을 추천한다.

In [ ]:
alco2009 = pd.read_csv("niaaa-report2009.csv", index_col="State")
alco2009

> **넓은 꼴(wide form)**에서 **긴 꼴(long form)**로 자료를 바꾸어보자. 인덱스로 주(`state`)가 설정된 상황이다. `stack()` 함수를 통해 넓은 꼴로 주어진 자료를 쌓아 길게 만들어줄 수 있다(Why?).

In [ ]:
wide2long = alco2009.stack()
wide2long

> 그리고나서 `unstack()`으로 다시 원래 형태, 즉 넓은 꼴로 되돌아갈 수 있다.

In [ ]:
wide2long.unstack()

> **연습문제 3-1**. 다음 `myjson`을 데이터프레임으로 변환하고, 이 자료를 이름별로 긴 꼴 변환하시오.
---
```python
rec1 = {"id": 1001,
      "name": "김전일",
      "motto" : "할아버지의 이름을 걸고"}
rec2 = {"id": 1002,
      "name": "코난",
      "motto" : "진실은 언제나 하나"}
rec3 = {"id" : 1003,
      "name" : "김현우",
      "motto" : "할많하않"}
data = [rec1, rec2, rec3]      
```

In [ ]:
rec1 = {"id": 1001,
      "name": "김전일",
      "motto" : "할아버지의 이름을 걸고"}
rec2 = {"id": 1002,
      "name": "코난",
      "motto" : "진실은 언제나 하나"}
rec3 = {"id" : 1003,
      "name" : "김현우",
      "motto" : "할많하않"}
data = [rec1, rec2, rec3]

df = pd.DataFrame(data)
df = df.set_index('name')
df
df.stack()

> pandas에서는 보다 어렵고 복잡한 자료 재배열을 효율적으로 수행하기 위해 여러 명령어를 지원한다. 본격적인 실무나 연구에서 사용하려면 (제일 꼼꼼하게 작동하는) `pivot()`이 유용할 수도 있다. 이 함수로 자료의 꼴을 재배열할 때는 새로운 행과 열을 지정하고, 각각의 행렬 안에 들어갈 원소를 지정하는 방식을 취한다.

In [ ]:
alco = pd.read_csv("niaaa-report.csv")
alco

alco.pivot(index = "Year", columns = "State", values = "Wine")

> **연습문제 3-2**. `penguins.csv`를 활용하여 펭균의 종별(Adelie, Chinstrap, Gentoo)로 수컷-암컷 간 평균 몸무게 차이를 출력하시오. 어떤 종에서 암수 몸무게 차이가 가장 큰가?

In [ ]:
df = pd.read_csv('penguins.csv')
bm = df.groupby(['species', 'sex'])['body_mass_g'].mean()
bm2 = bm.unstack()
bm2['mass_diff'] = bm2['Male'] - bm2['Female']
bm2

## 4. 재부호화

> 자료 내용 자체를 건드릴 일은 흔하지 않을 거라고 생각하기 쉽다. 그러나 연구 분야에 따라서는 자료를 수정하거나 **재부호화(recoding)**하는 경우가 매우 잦다.  이때 특히 `replace()`를 활용할 때가 많다.
>
> 임의의 독일인에 관한 나이, 직업, 성별, 주택 보유 여부, 만기, 부채 목적, 부채액수 등을 담은 자료를 사용하자.

In [ ]:
#자료 불러오기
df = pd.read_csv('German_credit.csv')
df
df['Sex'].value_counts()

#변경
changes = {'male': 'M',
           'female': 'F'}
df['gender'] = df['Sex'].replace(changes)
df

> 이번엔 연속변수인 나이(`Age`)를 구간별 자료로 바꾸어 보자. 이때 `cut()`을 사용한다.

In [ ]:
df['Age'].describe()

# cut 메서드를 이용한 수치형 변수 구간화
agecat = pd.cut(df['Age'], bins=5)
agecat
agecat.value_counts()

> 이때 `bins=5`같은 옵션은 정확히 구간을 나눌때 불편할 수 있다. 이 경우 리스트(list)를 옵션으로 제공하면 된다. 10부터 20까지, ..., 70부터 80까지 나타낸다. 이때 `()`는 **열린 괄호(open bracket)**로 초과미만을, `[]`는 **닫힌 괄호(close bracket)**로 이상이하를 뜻한다.

In [ ]:
# 디폴트
agecat = pd.cut(df['Age'], bins=[10, 20, 30, 40, 50, 60, 70, 80])
agecat
agecat.value_counts()

# right 인자를 이용한 오른쪽 닫힌 구간 범주화
agecat = pd.cut(df['Age'], bins= [10, 20, 30, 40, 50, 60, 70, 80], right=False)
agecat
agecat.value_counts()

> **범주형 변수(categorical variables)**를 분석하려면 때때로 **가변수(dummy variables)**로 **가부호화(dummy coding)**해야 할 필요가 있다. 이때는 `pd.get_dummies()` 함수를 사용한다.

In [ ]:
pd.get_dummies(df['island'])

> 새로 생겨야 하는 변수는 3개이므로 새로운 컬럼 이름도 3개를 지정해야 한다(Why?).

In [ ]:
df[['island1', 'island2', 'island3']] = pd.get_dummies(df['island'])
df

> 아니면 아예 `pd.concat()`을 사용해 결합한다!

In [ ]:
df = pd.read_csv('penguins.csv')
islands = pd.get_dummies(df['island'])
df = pd.concat([df, islands], axis = 1)
df

> **연습문제 4-1**. `German_credit.csv`에서 부채액(`Credit amount`)을 십분위수로 나누고 원자료에 새로 결합하시오.

In [ ]:
#자료 불러오기
df = pd.read_csv('German_credit.csv')
tba = pd.cut(df['Credit amount'], bins=10)
pd.concat([df, tba], axis = 1)

> **연습문제 4-2**. `division.csv`와 `niaaa-report.csv`를 결합하고 지역(`division`)을 범주형 변수로 재부호화하시오.

In [ ]:
import pandas as pd

division = pd.read_csv("division.csv")
tba = pd.get_dummies(division['region'])
df1 = pd.concat([division, tba], axis =1)

alco = pd.read_csv("niaaa-report.csv")
df2 = pd.merge(alco, df1, left_on='State', right_on='state')
df2

## 5. 자료 전처리 연습

> 자료 전처리를 실제로 처음부터 끝까지 제대로 수행해보자. 이번에 사용할 자료 `Uber.csv`는 미국, 스리랑카, 파키스탄의 2016년 1월부터 12월까지 우버 운행 기록이다.

In [ ]:
import gdown
link = 'https://drive.google.com/uc?id=1csKImQToC6_JTRbyan9aJuDq9UYHw4tr'
gdown.download(link)

In [ ]:
df = pd.read_csv('Uber.csv')
df

> 다양한 변수가 있는데 훑어보는 것도 중요하다!

In [ ]:
df.info()
df.describe()
df['CATEGORY'].value_counts()
df['PURPOSE'].value_counts()
df['START'].unique()
df['STOP'].unique()

> 앞서 `pd.to_datetime()`을 통해 object를 날짜 속성으로 바꿀 수 있음을 배웠다. 출발과 도착 시간을 먼저 가공하자. 가공하고 살펴보면 편리하게도 날짜와 시간 출력 형식이 모두 일관되도록 수정되었다. 또 예측치 못한 에러(error)가 나오더라도 무시하고 강제로 건너뛰었다(Why?).

In [ ]:
df['start'] = pd.to_datetime(df['START_DATE'], errors='coerce')
df['end'] = pd.to_datetime(df['END_DATE'], errors='coerce')
df

> `sort_values()`를 사용한다면 자료를 정렬(sort)할 수도 있다.

In [ ]:
df.sort_values(['start','end'])

> 우버 사용 시간이 어떻게 되는지 계산해보자.

In [ ]:
# datetime 변수의 단위 변환 (분)
df['duration'] = (df['end'] - df['start'])
df

#확인
df['duration'].sort_values()

> 주행시간을 살펴보니 벌써 **이상치(outliers)** 3개가 눈에 들어온다.

In [ ]:
outlier_idx = [269, 776, 1155]
df.loc[outlier_idx]

#drop()으로 삭제
df.drop(outlier_idx, axis = 0)

> 이제 `CATEGORY`와 `PURPOSE` 별로 이용 시간(`duration`)의 평균을 비교해보자.

In [ ]:
df.groupby(['CATEGORY', 'PURPOSE'])['duration'].mean()